# Variance Reduction Comparison: No Covariates vs Raw Covariates vs CUPAC vs MLRATE

In this example we compare four ways of handling a covariate `x` that is **non-linearly**
related to the outcome `target`:

1. **No covariates** — plain difference-in-means, `x` is ignored entirely.
2. **Raw covariate (no ML)** — add `x` directly as a regression covariate, no model involved.
3. **CUPAC** — fit a model on **pre-experiment data** to predict `target` from `x`, then use
   the prediction as a covariate on experiment data.
4. **MLRATE** — K-fold cross-fit a model **on the experiment data itself**
   ([Guo et al., NeurIPS 2021](https://arxiv.org/abs/2106.07263)) to predict `target` from `x`;
   no pre-experiment data required.

Because the relationship is non-linear, a raw linear covariate barely helps — but CUPAC and
MLRATE can plug in a gradient-boosted model instead, which captures it well. We hold the
effect size and simulation count fixed across all four and compare the resulting power.

In [ ]:
import numpy as np
import pandas as pd
import plotnine as p9
from sklearn.ensemble import HistGradientBoostingRegressor

from cluster_experiments import (
    ConstantPerturbator,
    NonClusteredSplitter,
    OLSAnalysis,
    PowerAnalysis,
)

### Data generation

`target` is a **non-linear** function of `x` — `2 * sin(2x) + x^2` — plus noise. A linear
covariate can only pick up the (small) linear component of this relationship, while a
gradient-boosted model can capture the curve directly.

We keep a **pre-experiment** slice — only needed by CUPAC, to fit its model — and an
**experiment** slice used by all four methods for the actual power simulation.

In [ ]:
np.random.seed(2026)

N = 4_000
x = np.random.normal(size=N)
target = 2 * np.sin(2 * x) + x**2 + np.random.normal(scale=1.0, size=N)
df = pd.DataFrame({"x": x, "target": target})

is_pre_experiment = np.random.rand(N) < 0.4
df_pre = df[is_pre_experiment].reset_index(drop=True)
df_analysis = df[~is_pre_experiment].reset_index(drop=True)

print(f"{len(df_pre) = }, {len(df_analysis) = }")
df_analysis.head()

### 1. No covariates

Plain difference-in-means: `x` is not used at all.

In [ ]:
perturbator = ConstantPerturbator(average_effect=0.15)
splitter = NonClusteredSplitter()

pw_none = PowerAnalysis(
    perturbator=perturbator,
    splitter=splitter,
    analysis=OLSAnalysis(),
    n_simulations=200,
    seed=2026,
)
power_none = pw_none.power_analysis(df_analysis)
print(f"No covariates: {power_none = }")

### 2. Raw covariate, no ML

`x` is included directly as a regression covariate. No model is fit anywhere — since OLS can
only capture a linear relationship, and `x`'s relationship with `target` is mostly non-linear
(`sin(2x) + x^2`), this barely helps.

In [ ]:
pw_raw = PowerAnalysis(
    perturbator=perturbator,
    splitter=splitter,
    analysis=OLSAnalysis(covariates=["x"]),
    n_simulations=200,
    seed=2026,
)
power_raw = pw_raw.power_analysis(df_analysis)
print(f"Raw covariate (no ML): {power_raw = }")

### 3. CUPAC

Fit a `HistGradientBoostingRegressor` on **pre-experiment data** (`df_pre`) to predict
`target` from `x`, then use that prediction (`estimate_target`) as the regression covariate
on experiment data. Unlike a linear covariate, a GBM can pick up the `sin(2x) + x^2` shape.

In [ ]:
pw_cupac = PowerAnalysis(
    perturbator=perturbator,
    splitter=splitter,
    analysis=OLSAnalysis(covariates=["estimate_target"]),
    cupac_model=HistGradientBoostingRegressor(),
    features_cupac_model=["x"],
    n_simulations=200,
    seed=2026,
)
power_cupac = pw_cupac.power_analysis(df_analysis, df_pre)
print(f"CUPAC: {power_cupac = }")

### 4. MLRATE

K-fold cross-fit a `HistGradientBoostingRegressor` **on the experiment data itself** — each
row's prediction comes from a fold that never trained on it, so there's no pre-experiment
data requirement and no overfitting bias.

In [ ]:
pw_mlrate = PowerAnalysis(
    perturbator=perturbator,
    splitter=splitter,
    analysis=OLSAnalysis(covariates=["estimate_target"]),
    cupac_model=HistGradientBoostingRegressor(),
    ml_option="mlrate",
    features_cupac_model=["x"],
    n_simulations=200,
    seed=2026,
)
# Note: no pre_experiment_df passed in — MLRATE never needs it.
power_mlrate = pw_mlrate.power_analysis(df_analysis)
print(f"MLRATE: {power_mlrate = }")

### Comparison

In [ ]:
results = pd.DataFrame(
    {
        "method": ["No covariates", "Raw covariate\n(no ML)", "CUPAC", "MLRATE"],
        "power": [power_none, power_raw, power_cupac, power_mlrate],
    }
)
results

In [ ]:
(
    p9.ggplot(results, p9.aes(x="method", y="power"))
    + p9.geom_col(fill="#4a86e8")
    + p9.theme_minimal()
    + p9.labs(x="", y="Power", title="Power by variance-reduction strategy")
)

### Takeaways

- `target`'s relationship with `x` here is deliberately non-linear (`2*sin(2x) + x^2`), so a
  **raw linear covariate barely helps at all** — its power is close to using no covariate.
- CUPAC and MLRATE both plug a `HistGradientBoostingRegressor` into that same slot instead of
  a linear term, and both recover most of the available variance reduction as a result.
- CUPAC needed a separate pre-experiment sample to fit its model; MLRATE got the same
  benefit using **only** the experiment data, via cross-fitting — no `pre_experiment_df`
  required.
- The gap between "raw covariate" and "CUPAC/MLRATE" here is a direct illustration of *why*
  it's worth cross-fitting or pre-fitting an ML model instead of just tossing the covariate
  into OLS: OLS can't bend to fit a non-linear shape, a GBM can.